# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks

Before creating the baseline score, I tested two signals to evaluate whether the patterns in the data support potential action signals.

---

### Signal 1: Staleness / Refresh Signal (`days_since_last_update`)

**Hypothesis:** Pages that have not been updated for a longer duration (e.g., 91+ days) exhibit higher observed decline rates compared to fresher pages (0–30 days), providing descriptive support for a content refresh flag.

See the bucket table output in the code cell below.

**Verdict: MIXED**

*Explanation:* Pages unupdated for 91–180 days show an observed decline rate of **61.11%**, compared to **51.14%** for pages updated within 0–30 days (+9.97 percentage points higher decline rate for stale pages). Overall, pages with 91+ days since update have a 60.85% decline rate vs 51.20% for fresher pages (<91 days). However, the oldest bucket (181+ days) has a lower decline rate (47.13%) likely due to a smaller sample size (n=174) or evergreen content. Thus, the verdict is **MIXED**: while staleness does not guarantee decline, 91+ days provides a useful empirical threshold for human review.

---

### Signal 2: Search Position → Click-Through Rate (`avg_position` vs `ctr`)

**Hypothesis:** Pages ranking worse in Google search results (higher `avg_position`) receive lower click-through rates (`ctr`), supporting FlyRank's CTR-fix and search position logic.

See the bucket table output in the code cell below.

**Verdict: CONFIRMED**

*Explanation:* Across active search rankings, mean CTR strictly decreases as search position worsens: **2.71%** for Top 3, **0.65%** for Page 1 (positions 3.1–10), **0.32%** for Striking Distance (positions 10.1–20), **0.22%** for Pages 3–5 (positions 20.1–50), and **0.15%** for Deep positions (50+). Pages with `avg_position == 0` (1,205 rows) represent missing position data rather than rank zero and are explicitly grouped as "no position data". The pattern strictly confirms that search visibility correlates with user CTR.

---

### Section 1 Takeaway

Both signals provide useful descriptive evidence for decision-support:
1. **Staleness (`days_since_last_update >= 91`)** serves as a valid condition for identifying pages that may benefit from a content refresh review.
2. **Search Position / CTR** complements staleness by identifying pages where rank drops directly correlate with lower user CTR.

*Note:* Neither signal is causal or a guaranteed predictor of Google's algorithm. They serve as transparent features for human decision-support.

In [1]:
import pandas as pd
from pathlib import Path

# Load dataset from repository root
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), "Starter CSV not found — run this notebook from repository root."
df = pd.read_csv(DATA_PATH)

# Derive observed decline label from trend_direction for descriptive audit only
# (This label is never used as an input feature in the baseline score)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

print("==================================================")
print("SIGNAL 1: Staleness / Refresh Signal")
print("==================================================")

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

# Calculate page count (n), decline count, and decline rate (%)
signal1_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
    .rename(columns={"staleness_bucket": "bucket"})
)
signal1_table["decline_rate"] = (signal1_table["decline_rate"] * 100).round(2)

print("\nSignal 1 Bucket Table (Staleness vs Observed Decline Rate):")
print(signal1_table.to_string(index=False))
print("\nVerdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.")

print("\n==================================================")
print("SIGNAL 2: Search Position vs CTR")
print("==================================================")

# Create position buckets and handle avg_position == 0 explicitly
pos_df = df.copy()
pos_df["position_bucket"] = pd.cut(
    pos_df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["top 3 (1-3)", "page 1 (3.1-10)", "striking distance (10.1-20)", "pages 3-5 (20.1-50)", "deep (50+)"],
    include_lowest=False
)
pos_df["position_bucket"] = pos_df["position_bucket"].cat.add_categories(["no position data (0)"])
pos_df.loc[pos_df["avg_position"] == 0, "position_bucket"] = "no position data (0)"

# Calculate page count (n), mean CTR, and median CTR
signal2_table = (
    pos_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
    .rename(columns={"position_bucket": "bucket"})
)
signal2_table["mean_ctr"] = signal2_table["mean_ctr"].round(2)
signal2_table["median_ctr"] = signal2_table["median_ctr"].round(2)

print("\nSignal 2 Bucket Table (Search Position vs CTR %):")
print(signal2_table.to_string(index=False))
print("\nVerdict: CONFIRMED — Mean CTR strictly decreases as search position worsens (2.71% in Top 3 down to 0.15% in Deep positions).")


SIGNAL 1: Staleness / Refresh Signal

Signal 1 Bucket Table (Staleness vs Observed Decline Rate):
     bucket     n  decline_count  decline_rate
  0-30 days 20480          10473         51.14
 31-90 days   175            103         58.86
91-180 days  9171           5604         61.11
  181+ days   174             82         47.13

Verdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.

SIGNAL 2: Search Position vs CTR

Signal 2 Bucket Table (Search Position vs CTR %):
                     bucket     n  mean_ctr  median_ctr
                top 3 (1-3)  1141      2.71        0.00
            page 1 (3.1-10) 11842      0.65        0.16
striking distance (10.1-20)  7273      0.32        0.10
        pages 3-5 (20.1-50)  7225      0.22        0.03
                 deep (50+)  1314      0.15        0.00
       no position data (0)  1205      0.30        0.00

Verdict: CONFIRMED — Mean CTR strictly decreases as 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.